## **Содержание**

Знакомство с нейросетевыми фреймворками на площадках с доступными ресурсами GPU/TPU.

### Базовый код

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [3]:
batch_size = 128

train_dataloader = DataLoader(training_data, batch_size=batch_size, num_workers=8)
test_dataloader = DataLoader(test_data, batch_size=batch_size, num_workers=8)

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 64)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

### CPU/GPU

In [5]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [6]:
model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [9]:
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
print("Done!")

### TPU

Для запуска обучения на TPU совместно с torch необходимо устанавливать библиотеку torch-xla. При установке обращайте внимание на соответствия версий torch и torch-xla.

#### Single-Core обучение

In [10]:
import torch_xla.core.xla_model as xm

In [11]:
num_tpu_cores = xm.xrt_world_size()
print("Number of TPU cores available:", num_tpu_cores)

In [12]:
device = xm.xla_device()

In [13]:
model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [14]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        xm.mark_step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    print(f"batch {batch}: loss {loss.item()}")

In [15]:
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
print("Done!")

#### Multiple-Core обучение

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

In [17]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 64)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [18]:
SERIAL_EXEC = xmp.MpSerialExecutor()

def get_data():
    return datasets.MNIST(
        root="data",
        train=True,
        download=True,
        transform=transforms.ToTensor()
    )

In [19]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    model.train()
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        # This call synchronizes the optimizer step across TPU cores.
        xm.optimizer_step(optimizer)
        if batch_idx % 100 == 0:
            print(f"[Process {xm.get_ordinal()}] Batch {batch_idx} - Loss: {loss.item():.4f}")

In [20]:
def _mp_fn(index):
    # Print process information for debugging.
    print(f"Process {index} | World Size: {xm.xrt_world_size()} | Ordinal: {xm.get_ordinal()}")

    # Download the dataset only once.
    train_dataset = SERIAL_EXEC.run(get_data)

    # Create a distributed sampler to split the data across TPU cores.
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=xm.xrt_world_size(),
        rank=xm.get_ordinal(),
        shuffle=True
    )

    # Get the XLA device for this process.
    device = xm.xla_device()

    # Create the model on the XLA device.
    model = NeuralNetwork().to(device)

    # Create the DataLoader.
    train_loader = DataLoader(
        train_dataset,
        batch_size=128,  # Adjust as needed.
        sampler=train_sampler,
        num_workers=4  # Adjust based on your environment.
    )

    # Wrap the DataLoader to move data to the correct device.
    para_loader = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)

    # Set up optimizer and loss function.
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    epochs = 5
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1} starting on process {xm.get_ordinal()}")
        train_loop(para_loader, model, loss_fn, optimizer, device)
        print(f"Epoch {epoch + 1} done on process {xm.get_ordinal()}")

Обучение рекомендуется запускать как единый файл, а не набор ячеек:

In [21]:
if __name__ == '__main__':
    # Spawn one process per TPU core. For Colab TPU, this is typically 8.
    xmp.spawn(_mp_fn, args=(), start_method='spawn') # start_method='fork'

## **Pytorch Lightning**

Данный фреймворк позволяет обучать на различных устройствах (CPU, GPU, TPU), удобно конфигурируя параметр устройства.

In [22]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from torchvision import datasets

In [23]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 64)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [24]:
class MnistModule(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = NeuralNetwork()
        self.loss_func = nn.CrossEntropyLoss()

    def forward(self, x):
        # in lightning, forward defines the prediction/inference actions
        return self.model(x)

    def training_step(self, batch, batch_idx):
        # training_step defined the train loop.
        # It is independent of forward
        x, y = batch
        out = self.model(x)
        loss = self.loss_func(out, y)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
        return optimizer

In [25]:
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

batch_size = 128
# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)

In [26]:
# init model
model = MnistModule()

# most basic trainer, uses good defaults (auto-tensorboard, checkpoints, logs, and more)
trainer = pl.Trainer()
# trainer = pl.Trainer(accelerator="gpu", devices=1)
# trainer = pl.Trainer(accelerator="tpu", devices=[1])
# trainer = pl.Trainer(accelerator="tpu", devices=8)
trainer.fit(model, train_dataloader)